In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from matplotlib import pyplot as plt
import pandas as pd

df = pd.read_csv('../Resources/Overall_cleaned.csv')
#have to drop these or else df.dropna() will return nothing
df = df.drop(columns = [
    'Grade Recode (thru 2017)',
    'Derived Summary Grade 2018 (2018+)',
    'Grade Clinical (2018+)',
    'Grade Pathological (2018+)',
    'Tumor Size Summary (2016+)',
    'Regional nodes positive (1988+)'
])
df_raw = pd.read_csv('../Resources/Overall.csv')
#sample because there is not enough memory to compute the distance matrix for the entire dataset, we will sample a subset of the data
#dropna as linkage requires no missing values
df_sample = df.dropna().sample(n=3000, random_state=42)
df_sample

In [ ]:
df.head(300).to_excel('output.xlsx', index=False)

In [ ]:
df_raw['Total number of in situ/malignant tumors for patient'].value_counts()

In [ ]:
# covariates = [
#     'Regional nodes positive (1988+)',
#     'Tumor Size Over Time Recode (1988+)',
#     'Age recode with <1 year olds and 90+',
#     'SEER Combined Mets at DX-lung (2010+)',
#     'Regional nodes examined (1988+)',
#     'Sex',
#     'SEER Combined Mets at DX-bone (2010+)',
#     'Year of diagnosis',
#     'Marital status at diagnosis',
#     'Grade Recode (thru 2017)',
#     'Histology recode - broad groupings',
#     'SEER Combined Mets at DX-liver (2010+)',
#     'SEER Combined Mets at DX-brain (2010+)',
#     'Primary Site',
#     'Tumor Size Summary (2016+)',
#     'AYA site recode 2020 Revision',
#     'Rural-Urban Continuum Code',
#     'Site recode - rare tumors',
#     'Mets at DX-Other (2016+)',
#     'Diagnostic Confirmation',
#     'Survival months',
#     'Vital status recode (study cutoff used)'
# ]

# df_sample = df_sample[covariates]

# df_sample = df_sample.drop(['Tumor Size Over Time Recode (1988+)','Tumor Size Summary (2016+)'], axis=1)
# X_sample = df_sample.drop(['Survival months','Vital status recode (study cutoff used)'], axis=1)


In [ ]:
ranges = []
#use try and except to handle categorical features
for col in df_sample.columns:
    try:
        col_min = df_sample[col].min()
        col_max = df_sample[col].max()
        col_range = col_max - col_min
        ranges.append(col_range)
    except:
        ranges.append(1)


In [ ]:
ranges = [float(x) for x in ranges]
print(ranges)

In [ ]:
def gower(u, v, w=None):
    """
    u - represents a row of data in the dataframe
    v - represents a row of data in the dataframe
    range - represents the range of each covariate in the dataframe
    """
    #had to hardcode the range of each covariate because I was getting an error when trying to compute it from the dataframe
    ranges = [75.0, 1.0, 6.0, 1.0, 1.0, 1.0, 1.0, 2080.0, 1.0, 83.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 157.0, 157.0, 63.0, 19.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 6.0, 1.0, 1.0]
    distance = 0

    for i in range(len(u)):
        #try converting to float, if it fails then we have a categorical variable
        try:
            u[i] = float(u[i])
            v[i] = float(v[i])
            distance += abs(u[i] - v[i])/ranges[i]
        except:
            if u[i] != v[i]:
                distance += 1
    return 1/len(u) * (distance)


In [ ]:

#edit linkage to compute Gower distance instead
linkage_matrix = linkage(df_sample, method='average', metric=gower)
#need to check method='average' isn't causing problems 


In [ ]:
#Each sample starts as its own cluster
#Find the pair of clusters with the smallest distance
#Combine pairs of clusters into one cluster
#Repeat steps 2-3 until only one cluster remains
plt.figure(figsize=(10, 10))
dendrogram(linkage_matrix,orientation='right', leaf_font_size=2)
plt.xlabel('Gower Distance', fontsize=12)
plt.ylabel('Patient Sample', fontsize=12)
plt.title('Agglomerative Clustering of Patient Cohorts\n(Gower Distance, Average Linkage)', 
          fontsize=14)
plt.show()

In [ ]:
#further analysis of groups, cut the tree at 0.3

from scipy.cluster.hierarchy import fcluster

#cut the tree at 0.3
labels = fcluster(linkage_matrix, t=2.1, criterion='distance')
df_sample['cluster'] = labels


In [ ]:
def plot_clusters_by_covariates(df_sample, covariate_x, covariate_y):
    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(df_sample[covariate_x], 
                         df_sample[covariate_y], 
                         c=df_sample['cluster'], 
                         cmap='tab10', 
                         s=50, 
                         alpha=0.6)
    
    plt.xlabel(covariate_x)
    plt.ylabel(covariate_y)
    plt.title(f'{covariate_x} vs {covariate_y} by Cluster')
    plt.colorbar(scatter, label='Cluster')
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
import numpy as np

df_sample['cluster'].value_counts()

In [ ]:
#Get plot 20 random pairs

import random

#get all columns except 'cluster'
columns = [col for col in df_sample.columns if col != 'cluster']

#randomly choose 20 pairs
random.seed(42)
pairs = random.sample([(columns[i], columns[j]) 
                       for i in range(len(columns)) 
                       for j in range(i+1, len(columns))], 
                      k=20)

#plot each pair
for cov_x, cov_y in pairs:
    plot_clusters_by_covariates(df_sample, cov_x, cov_y)


In [ ]:
#Label encode categorical columns
#For getting cluster means and heatmap plots
from sklearn.preprocessing import LabelEncoder

#identify categorical columns
categorical_cols = df_sample.select_dtypes(include=['object', 'category']).columns

#;abel encode each categorical column
df_encoded = df_sample.copy()

for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))



In [ ]:
print(pd.Series(labels).value_counts().sort_index())

In [ ]:
#specialised heatmap

import seaborn as sns
# Normalize each feature and plot by cluster

cluster_means = df_encoded.groupby('cluster').mean()

#min max scaling, shift the bottom of the data to the 0 axis
cluster_means_normalized = (cluster_means - cluster_means.min(axis=0)) / (cluster_means.max(axis=0) - cluster_means.min(axis=0))

cluster_means_normalized_spec = cluster_means_normalized.loc[[1,3,4]]
cluster_means_normalized_spec = cluster_means_normalized_spec.drop('Behavior recode for analysis', axis=1)




In [ ]:
#split the features in half
num_features = len(cluster_means_normalized_spec.T)
mid_point = num_features // 2

# Top half of features
plt.figure(figsize=(3, 9))
sns.heatmap(cluster_means_normalized_spec.T[:mid_point], 
            cmap='RdYlGn', annot=True, fmt='.2f')
plt.title('Feature Profile by Cluster - Part 1')

plt.show()

# Bottom half of features
plt.figure(figsize=(3, 9))
sns.heatmap(cluster_means_normalized_spec.T[mid_point:], 
            cmap='RdYlGn', annot=True, fmt='.2f')
plt.title('Feature Profile by Cluster - Part 2')
plt.show()


In [ ]:
#calculate variance for each covariate across clusters
variance_by_covariate = cluster_means_normalized_spec.drop(['Survival months','Schema ID (2018+)'], axis=1).var().sort_values(ascending=False)

#get top 10
top_10_variance = variance_by_covariate.head(10)
print(top_10_variance)

#get just the column names
top_10_cols = top_10_variance.index.tolist()
print(top_10_cols)


In [ ]:
#get kmf probabilities of survival at the 60 month time point for each cluster
from lifelines import KaplanMeierFitter
kmf = KaplanMeierFitter()

clusters = [1,3,4]
probs = []
for i in clusters:
    cluster = df_sample[df_sample['cluster']==i]
    kmf.fit(durations=cluster['Survival months'].values, event_observed=cluster['Vital status recode (study cutoff used)'].values, label=f'Cluster {i}')
    survival_score = kmf.survival_function_at_times(times=[60]).values
    probs.append(survival_score[0])
    print(f'Cluster {i} Survival Probability at 60 months: {survival_score[0]:.4f}')

#sort by probs descending
sorted_indices = np.argsort(probs)[::-1]  # [::-1] reverses to get descending order

#apply same sorting to both
sorted_probs = np.array(probs)[sorted_indices]
sorted_clusters = np.array(clusters)[sorted_indices]

print(sorted_probs)
print(sorted_clusters)



In [ ]:
cluster_means_normalized_spec = pd.DataFrame(cluster_means_normalized_spec, columns=cluster_means_normalized_spec.columns, index=cluster_means_normalized_spec.index)
cluster_means_normalized_spec['Survival Probability at 60 months'] = pd.Series(probs, index=cluster_means_normalized_spec.index)

output_table = cluster_means_normalized_spec[['Survival Probability at 60 months']+top_10_cols]
output_table = output_table.loc[[1,3,4]]
output_table

In [ ]:
#reorder columns: Survival Probability first, then top 10 columns
all_cols = ['Survival Probability at 60 months'] + top_10_cols

#split columns in half
mid_point = len(all_cols) // 2

#first table: first half of columns
output_table_1 = cluster_means_normalized_spec[all_cols[:mid_point]]
output_table_1 = output_table_1.loc[[1,3,4]]

#second table: second half of columns (including survival probability for reference)
output_table_2 = cluster_means_normalized_spec[['Survival Probability at 60 months'] + all_cols[mid_point:]]
output_table_2 = output_table_2.loc[[1,3,4]]

print("Table 1:")
print(output_table_1)
print("\nTable 2:")
print(output_table_2)


In [ ]:
from sklearn.preprocessing import LabelEncoder

#fit LabelEncoder on the Grade Recode column
le = LabelEncoder()
le.fit(df['Grade Recode (thru 2017)'].astype(str))

#create mapping dictionary
grade_mapping = dict(zip(le.classes_, le.transform(le.classes_)))

#display the mapping
for category, number in sorted(grade_mapping.items(), key=lambda x: x[1]):
    print(f"{number:2d} → {category}")


In [ ]:
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter

#filter for cluster 4 patients
cluster_4_mask = df_sample['cluster'] == 4  # or however you defined cluster assignments
cluster_4_data = df_sample[cluster_4_mask].copy()

#get rural continuum code
rural_code = cluster_4_data['Race recode (W, B, AI, API)'].values

#create KM fitter for each rural code category and extract 12-month survival
kmf = KaplanMeierFitter()
survival_probs = []
rural_codes_unique = []

for code in cluster_4_data['Race recode (W, B, AI, API)'].unique():
    if pd.isna(code):
        continue
    
    # Fit KM for this rural code
    mask = cluster_4_data['Race recode (W, B, AI, API)'] == code
    kmf.fit(cluster_4_data[mask]['Survival months'],
            event_observed=cluster_4_data[mask]['Vital status recode (study cutoff used)'],
            label=f'Rural Code: {code}')
    
    # Get survival probability at 12 months
    surv_at_12 = kmf.survival_function_at_times(12).values[0]
    survival_probs.append(surv_at_12)
    rural_codes_unique.append(code)

#create scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(rural_codes_unique, survival_probs, s=100, alpha=0.6, color='steelblue')
plt.xlabel('Race recode (W, B, AI, API)', fontsize=12)
plt.ylabel('Survival Probability at 12 months', fontsize=12)
plt.title('Race recode vs 12-Month Survival Probability\n(Cluster 4 Patients)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.ylim([0, 1])
plt.tight_layout()
plt.show()
#jump beyond 68k income point implies that people that earn more than this amount increase their chances of survival by 10%
#


In [ ]:
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter

#filter for cluster 3 patients
cluster_3_mask = df_sample['cluster'] == 3  # or however you defined cluster assignments
cluster_3_data = df_sample[cluster_3_mask].copy()

#get rural continuum code
rural_code = cluster_3_data['Marital status'].values

#create KM fitter for each rural code category and extract 12-month survival
kmf = KaplanMeierFitter()
survival_probs = []
rural_codes_unique = []

for code in cluster_3_data['Sex'].unique():
    if pd.isna(code):
        continue
    
    # Fit KM for this rural code
    mask = cluster_3_data['Sex'] == code
    kmf.fit(cluster_3_data[mask]['Survival months'],
            event_observed=cluster_3_data[mask]['Vital status recode (study cutoff used)'],
            label=f'Rural Code: {code}')
    
    # Get survival probability at 12 months
    surv_at_12 = kmf.survival_function_at_times(12).values[0]
    survival_probs.append(surv_at_12)
    rural_codes_unique.append(code)

#create scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(rural_codes_unique, survival_probs, s=100, alpha=0.6, color='steelblue')
plt.xlabel('Sex', fontsize=12)
plt.ylabel('Survival Probability at 12 months', fontsize=12)
plt.title('Sex vs 12-Month Survival Probability\n(Cluster 3 Patients)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.ylim([0, 1])
plt.tight_layout()
plt.show()
#jump beyond 68k income point implies that people that earn more than this amount increase their chances of survival by 10%


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

#filter for cluster 4 patients with events (Vital status == 1)
cluster_4_mask = df_sample['cluster'] == 4
event_mask = df_sample['Vital status recode (study cutoff used)'] == 1
cluster_4_events = df_sample[cluster_4_mask & event_mask].copy()

#create scatter plot
plt.figure(figsize=(10, 6))
plt.scatter(cluster_4_events['Median household income inflation adj to 2023'], 
            cluster_4_events['Survival months'], 
            s=100, alpha=0.6, color='steelblue')
plt.xlabel('Median Household Income (Inflation-Adjusted to 2023)', fontsize=12)
plt.ylabel('Survival Months', fontsize=12)
plt.title('Median Household Income vs Survival Months\n(Cluster 4 Patients with Events)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

#filter for cluster 4 patients with events
cluster_4_mask = df_sample['cluster'] == 3
event_mask = df_sample['Vital status recode (study cutoff used)'] == 1
cluster_4_events = df_sample[cluster_4_mask & event_mask].copy()

#group by income and calculate mean survival months
grouped = cluster_4_evesnts.groupby('Median household income inflation adj to 2023')['Survival months'].agg(['mean', 'count'])

plt.figure(figsize=(10, 6))
#bubble size represents number of patients in that income group
plt.scatter(grouped.index, grouped['mean'], s=grouped['count']*5, alpha=0.6, color='steelblue')
plt.xlabel('Median Household Income (Inflation-Adjusted to 2023)', fontsize=12)
plt.ylabel('Mean Survival Months', fontsize=12)
plt.title('Median Household Income vs Mean Survival Months\n(Cluster 4 Patients with Events)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

#if you have the encoder saved from your label encoding step
#recreate it if needed:
le = LabelEncoder()
le.fit(df_sample['Primary Site - labeled'].astype(str))

#create a mapping DataFrame
mapping_df = pd.DataFrame({
    'Original Value': le.classes_,
    'Encoded Value': le.transform(le.classes_)
})

print(mapping_df)


In [ ]:
print(df_sample['Primary Site - labeled'])

In [ ]:

df_sample['Tumor Size Adjusted'] = np.where(
    df_sample['Primary Site - labeled'] == 'C34.1-Upper lobe,lung',
    df_sample['Total number of in situ/malignant tumors for patient'] * 1,
    np.where(
        df_sample['Primary Site - labeled'] == 'C34.2-Middle lobe',
        df_sample['Total number of in situ/malignant tumors for patient'] * 4,
        np.where(
            df_sample['Primary Site - labeled'] == 'C34.3-Lower lobe,lung',
            df_sample['Total number of in situ/malignant tumors for patient'] * 2,
            np.where(
                df_sample['Primary Site - labeled'] == 'C34.9-Lung, NOS',
                df_sample['Total number of in situ/malignant tumors for patient'] * 4,
                np.where(
                    df_sample['Primary Site - labeled'] == 'C34.0-Main bronchus',
                    df_sample['Total number of in situ/malignant tumors for patient'] * 5,
                    df_sample['Total number of in situ/malignant tumors for patient'] * 6
                )
            )
        )
    )
)


In [ ]:
df_sample['Primary Site - labeled'].value_counts()

In [ ]:
#Perform Likelihood Ratio Test to determine impact of 'Tumor Size Adjusted' covariate
import pandas as pd
from lifelines import CoxPHFitter
from scipy.stats import chi2

covariates = [
    'Primary Site - labeled',
    'Total number of in situ/malignant tumors for patient',
    'Tumor Size Adjusted',
    'Survival months',
    'Vital status recode (study cutoff used)'
    ]

df_clean = df_sample[covariates].dropna()

#null model - primary site and tumor number only
X_null = df_clean[['Primary Site - labeled', 'Total number of in situ/malignant tumors for patient']].copy()
#so that models can interpret the values for primary site
X_null = pd.get_dummies(X_null, columns=['Primary Site - labeled'], drop_first=True)

fit_data_null = X_null.copy()
fit_data_null['Survival months'] = df_clean['Survival months'].values
fit_data_null['Vital status recode (study cutoff used)'] = df_clean['Vital status recode (study cutoff used)'].values

cph_null = CoxPHFitter()
cph_null.fit(fit_data_null, duration_col='Survival months', event_col='Vital status recode (study cutoff used)')

print("NULL MODEL RESULTS:")
print(cph_null.summary)
print(f"\nLog-Likelihood: {cph_null.log_likelihood_:.4f}")
print(f"Concordance Index: {cph_null.concordance_index_:.4f}")

#alternative model - primary site, tumor number, and tumor size adjusted
X_alt = df_clean[['Primary Site - labeled', 'Total number of in situ/malignant tumors for patient', 'Tumor Size Adjusted']].copy()
X_alt = pd.get_dummies(X_alt, columns=['Primary Site - labeled'], drop_first=True)

fit_data_alt = X_alt.copy()
fit_data_alt['Survival months'] = df_clean['Survival months'].values
fit_data_alt['Vital status recode (study cutoff used)'] = df_clean['Vital status recode (study cutoff used)'].values


cph_alt = CoxPHFitter()
cph_alt.fit(fit_data_alt, duration_col='Survival months', event_col='Vital status recode (study cutoff used)')

print("\n" + "="*80)
print("ALTERNATIVE MODEL RESULTS:")
print(cph_alt.summary)
print(f"\nLog-Likelihood: {cph_alt.log_likelihood_:.4f}")
print(f"Concordance Index: {cph_alt.concordance_index_:.4f}")



In [ ]:
from scipy.stats import chi2

#extract log-likelihoods from your models
ll_null = -11061.1213
ll_alt = -11057.9892

#calculate LRT statistic
lrt_statistic = -2 * (ll_null - ll_alt)

#degrees of freedom = number of additional parameters
#null model: 6 covariates (tumor count + 5 Primary Site dummies)
#alternative model: 7 covariates (above + Tumor Size Adjusted)
df = 1

#calculate p-value
p_value = 1 - chi2.cdf(lrt_statistic, df)

print(f"Null Model Log-Likelihood: {ll_null:.4f}")
print(f"Alternative Model Log-Likelihood: {ll_alt:.4f}")
print(f"LRT Statistic: {lrt_statistic:.4f}")
print(f"Degrees of Freedom: {df}")
print(f"P-value: {p_value:.6f}")

#interpretation
alpha = 0.05
if p_value < alpha:
    print(f"\nRESULT: REJECT null hypothesis (p = {p_value:.6f} < {alpha})")
    print("Tumor Size Adjusted SIGNIFICANTLY improves the model")
else:
    print(f"\nRESULT: FAIL TO REJECT null hypothesis (p = {p_value:.6f} >= {alpha})")
    print("Tumor Size Adjusted does NOT significantly improve the model")


In [ ]:
print(df_clean['Tumor Size Adjusted'].value_counts())

In [ ]:
#simple null model with just one variable
cph_simple = CoxPHFitter()
cph_simple.fit(fit_data_null, duration_col='Survival months', 
               event_col='Vital status recode (study cutoff used)')
print(cph_simple.summary)

#now try alternative with just Tumor Size Adjusted (no tumor count)
X_alt_simple = df_clean[['Tumor Size Adjusted']].copy()
fit_data_alt_simple = X_alt_simple.copy()
fit_data_alt_simple['Survival months'] = df_clean['Survival months'].values
fit_data_alt_simple['Vital status recode (study cutoff used)'] = df_clean['Vital status recode (study cutoff used)'].values

cph_alt_simple = CoxPHFitter()
cph_alt_simple.fit(fit_data_alt_simple, duration_col='Survival months', 
                   event_col='Vital status recode (study cutoff used)')
print(cph_alt_simple.summary)
df_cph_alt_simple = pd.DataFrame(cph_alt_simple.summary)
print(df_cph_alt_simple)

In [ ]:
df_cph_alt_simple